- https://github.com/vllm-project/vllm/blob/main/docs/configuration/optimization.md
- https://www.youtube.com/watch?v=HJUte-vwQHo

在 vLLM 的底层架构中，显存（VRAM）分配是一场“中间激活值（Activation Memory）”与“全局缓存（KV Cache）”之间的零和博弈。

vLLM 在启动时，会根据设定的参数进行一次显存性能分析（Memory Profiling）：它会进行一次虚拟的前向传播（Dummy Forward Pass），测算出极端情况下“激活值”所需的显存峰值，然后将剩余的所有显存划分给 PagedAttention 用于构建 KV Cache Pool。

在 vLLM 引擎初始化的最后一步（你可以查看 vLLM 源码的 worker/model_runner.py 中的 profile_run 函数），为了计算出到底能留多少显存给 KV Cache，vLLM 会进行一次 Dummy Profiling（虚拟试运行）。它的源码逻辑大致如下：

- 它会根据配置，构造一个虚拟的 Prefill 批次（Dummy Batch）。
- 这个批次里包含 max_num_seqs 个虚拟请求。
- 它会强行将每个请求的长度拉长，使得所有请求的 Token 数量之和精确等于 max_num_batched_tokens。
    - (例如，配置了 max_num_seqs=32，max_num_batched_tokens=8192，它就会模拟同时塞入 32 个请求，每个请求长度为 256)。
- 所有的 Token 经过 Continuous Batching 组合成一个形状为 `[max_num_batched_tokens]` 的 1D 张量，扔进模型跑一次真实的前向传播。
- 通过 torch.cuda.max_memory_allocated() 捕获这期间瞬间飙升的显存峰值（这就代表了未来系统可能面临的最坏情况）。
- 最后，用总显存减去模型权重显存和这个“峰值激活显存”，剩下的全部切碎分给 KV Cache Block。

```
[ GPU VRAM Allocation in vLLM ]
|===========================================================|
| 1. Model Weights (Fixed by model size / precision)        |
|-----------------------------------------------------------|
| 2. Activation Memory (Determined by parameters below)     |
|    - Prefill Peak ∝ max_num_batched_tokens                |
|    - Decode Peak  ∝ max_num_seqs                          |
|    - Sequence Cap = max_model_len                         |
|-----------------------------------------------------------|
| 3. KV Cache Pool (The remaining memory)                   |
|    - Size = Total_VRAM - (Weights + Activation_Peak)      |
|===========================================================|
```

### 核心参数

- max_model_len：单条数据（Prompt + 自动生成的 Response）允许的最大 Token 数量。
    - 激活值爆炸风险：如果不开启 Chunked Prefill，单条长序列在 Prefill（预填充）阶段必须一次性处理完。如果 max_model_len 设得极大（例如 32k），即使你实际传入的序列只有 2k，系统也需要预留庞大的显存以防万一，这会极大地挤占 KV Cache 的空间。
- max_num_batched_tokens：单批次最大 Token 数
    - verl config：`max_num_batched_tokens: 8192`
    - vLLM 使用了 FlashAttention 和 连续批处理（Continuous Batching）。
        - FlashAttention 抹平了二次方显存：在传统 Transformer 中，由于需要在 GPU 显存（HBM）中实例化完整的 $N \times N$ 注意力分数矩阵，你的计算是成立的。但 FlashAttention 是在 GPU 的片上高速缓存（SRAM）里分块计算 Attention，绝不会在显存中生成那个 $O(N^2)$ 的庞大矩阵。
- max_num_seqs：
    - 单次前向传播中，能够同时并发处理的独立请求（Sequence）的最大数量。它主要约束 Decode（解码）阶段。
    - 决定 Decode 激活值峰值：在生成回复的 Decode 阶段，每个序列每次只吐出 1 个 Token，此时的 Batch Size 其实就是并发的序列数。vLLM 会模拟 Batch Size = max_num_seqs 进行显存测算。
    - 并发调度上限：这个参数等于系统向调度器承诺的“最大并发槽位”。设得越高，调度器会尝试把越多的请求塞入 GPU，这也意味着你需要更大的 KV Cache Pool 来支撑这 32 个序列的上下文。如果显存不足，多余的请求就会被挂起（Pending）。

### qwen2.5-vl-3b 为例

- weights
    - Qwen2.5-VL-3B 参数量：约 30 亿（LLM）+ 视觉编码器，总计约 32 亿参数。
    - 精度：使用 BF16/FP16，每个参数占 2 Bytes。
    - 消耗计算：$3.2 \times 10^9 \times 2 \text{ Bytes} \approx \mathbf{6.4 \text{ GB}}$
- 单 Token 的 KV Cache 成本
    - KV Cache 的大小不取决于序列多长，而是看你存了多少个 Token。我们先算出一个 Token 的 KV Cache 有多大。
    - 计算公式：$2 \text{ (K和V)} \times \text{层数}(L) \times \text{KV头数}(N_{kv}) \times \text{单头维度}(D) \times 2 \text{ Bytes}$
    - 3B 模型典型架构 (GQA)：层数 $L=36$，KV 头数 $N_{kv}=4$，单头维度 $D=128$。
    - 消耗计算：$2 \times 36 \times 4 \times 128 \times 2 \text{ Bytes} = \mathbf{73,728 \text{ Bytes} \approx 72 \text{ KB / Token}}$
- 激活值 (Activation Peak)
    - 这就是 max_num_batched_tokens 发挥作用的地方。在推理的前向传播中，不需要保存反向传播的梯度图，最大的张量通常发生在 MLP (全连接层) 展开的瞬间。
    - 经验估算：在最宽的隐藏层处，每个 Token 的中间激活状态大约占用 40~60 KB（具体取决于膨胀系数）。我们保守取 50 KB / Token。
        - $\approx 8192 \text{ tokens} \times 50 \text{ KB} \approx \mathbf{400 \text{ MB} (0.4 \text{ GB})}$。